# APAN 5200 Machine Learning — Non-linear Regression (Mock Quiz)
## Practice Questions
**Dataset:** Spotify Popular Songs  
**Outcome Variable:** `rating`

Non-linear regression extends linear regression by allowing curved, flexible relationships
between predictors and the outcome. Key methods covered:
- **Polynomial Regression** — add higher-order terms (x², x³, …)
- **KNN Regression** — predict using the average of k nearest neighbors

In [ ]:
# ============================================================
# VARIABLE DESCRIPTIONS (same dataset as Q8)
# ============================================================
# 0.  id                : Song ID (dropped)
# 1.  performer         : Performer name (dropped)
# 2.  song              : Song name (dropped)
# 3.  genre             : Genre (dropped)
# 4.  track_duration    : Duration in milliseconds
# 5.  track_explicit    : Is explicit
# 6.  danceability      : How suitable for dancing (0.0–1.0)
# 7.  energy            : Perceptual intensity/activity (0.0–1.0)
# 8.  key               : Estimated key of track (dropped)
# 9.  loudness          : Overall loudness in dB
# 10. mode              : Major (1) or Minor (0)
# 11. speechiness       : Presence of spoken words (0.0–1.0)
# 12. acousticness      : Confidence of acoustic track (0.0–1.0)
# 13. instrumentalness  : Likelihood of no vocals (0.0–1.0)
# 14. liveness          : Presence of audience (0.0–1.0)
# 15. valence           : Musical positiveness (0.0–1.0)
# 16. tempo             : Beats per minute
# 17. time_signature    : Time signature
# 18. rating            : TARGET
# 19–28. genre_*        : Ten binary genre dummy variables

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

In [ ]:
# ============================================================
# Q1: Data Setup & Stratified Split
#
# Read the data. Drop: id, performer, song, genre, key.
# Separate y = rating, X = remaining features.
# Create y_binned = pd.qcut(data.rating, q=20)
# Stratified split: 70% train / 30% test, random_state=1031
#
# QUESTION: How many features (columns) does X have after
#           dropping the specified variables?
# ANSWER  : 23 features
# ============================================================

data = pd.read_csv('spotify_data.csv')
data = data.drop(columns=['id', 'performer', 'song', 'genre', 'key'])

y = data['rating']
X = data.drop(columns=['rating'])

y_binned = pd.qcut(data.rating, q=20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1031, stratify=y_binned
)

print(f'Number of features in X: {X.shape[1]}')
print(f'Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}')

In [ ]:
# ============================================================
# Q2: Linear Regression Baseline — Single Feature (loudness)
#
# Construct a simple linear regression using ONLY loudness
# to predict rating.
#
# QUESTION: What is the RMSE in the TRAIN sample?
# ANSWER  : 16.2925
#
# KEY CONCEPT: This is our baseline. Non-linear models should
# outperform this if there is a non-linear relationship between
# loudness and rating.
# ============================================================

lr_base = LinearRegression()
lr_base.fit(X_train[['loudness']], y_train)

rmse_lr_train = np.sqrt(mean_squared_error(y_train, lr_base.predict(X_train[['loudness']])))
rmse_lr_test  = np.sqrt(mean_squared_error(y_test,  lr_base.predict(X_test[['loudness']])))

print(f'Linear Regression (loudness) RMSE — Train: {rmse_lr_train:.4f}')
print(f'Linear Regression (loudness) RMSE — Test : {rmse_lr_test:.4f}')

In [ ]:
# ============================================================
# Q3: Polynomial Regression (degree=2) — Single Feature
#
# Fit a Polynomial Regression of degree 2 using ONLY loudness.
# Use Pipeline([PolynomialFeatures(degree=2), LinearRegression()]).
#
# QUESTION: What is the RMSE in the TRAIN sample?
# ANSWER  : 16.2617
#
# KEY CONCEPT: Degree-2 polynomial adds loudness² as a feature,
# capturing quadratic (U-shaped) relationships. The slight
# improvement over degree-1 suggests weak non-linearity.
# ============================================================

poly2 = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('lr',   LinearRegression())
])
poly2.fit(X_train[['loudness']], y_train)

rmse_p2_train = np.sqrt(mean_squared_error(y_train, poly2.predict(X_train[['loudness']])))
rmse_p2_test  = np.sqrt(mean_squared_error(y_test,  poly2.predict(X_test[['loudness']])))

print(f'Poly(degree=2) RMSE — Train: {rmse_p2_train:.4f}')
print(f'Poly(degree=2) RMSE — Test : {rmse_p2_test:.4f}')

In [ ]:
# ============================================================
# Q4: Polynomial Regression (degree=5) — Single Feature
#
# Fit a Polynomial Regression of degree 5 using ONLY loudness.
#
# QUESTION: What is the RMSE in the TRAIN sample?
# ANSWER  : 16.2354
#
# KEY CONCEPT: As degree increases, training RMSE decreases
# slowly, but the risk of OVERFITTING grows. Monitor test RMSE
# to spot the inflection point.
# ============================================================

poly5 = Pipeline([
    ('poly', PolynomialFeatures(degree=5, include_bias=False)),
    ('lr',   LinearRegression())
])
poly5.fit(X_train[['loudness']], y_train)

rmse_p5_train = np.sqrt(mean_squared_error(y_train, poly5.predict(X_train[['loudness']])))
rmse_p5_test  = np.sqrt(mean_squared_error(y_test,  poly5.predict(X_test[['loudness']])))

print(f'Poly(degree=5) RMSE — Train: {rmse_p5_train:.4f}')
print(f'Poly(degree=5) RMSE — Test : {rmse_p5_test:.4f}')

In [ ]:
# ============================================================
# Q5: Polynomial Regression (degree=2) — All Features
#
# Fit a degree-2 Polynomial Regression using ALL predictors.
#
# QUESTION A: How many features are created after applying
#             PolynomialFeatures(degree=2) to all 23 features?
# ANSWER A  : 300 features  (= 1 + 23 + 23*24/2 = 300)
#
# QUESTION B: What is the RMSE in the TRAIN sample?
# ANSWER B  : 15.2410
#
# KEY INSIGHT: Adding interaction terms between all features
# substantially reduces RMSE compared to single-feature polynomial.
# But dimensionality explodes — 23 features → 300.
# ============================================================

poly2_all = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('lr',   LinearRegression())
])
poly2_all.fit(X_train, y_train)

# Number of features
pf = PolynomialFeatures(degree=2, include_bias=True)
pf.fit(X_train)
print(f'Number of features after Poly(degree=2): {pf.transform(X_train).shape[1]}')

rmse_p2_all_train = np.sqrt(mean_squared_error(y_train, poly2_all.predict(X_train)))
rmse_p2_all_test  = np.sqrt(mean_squared_error(y_test,  poly2_all.predict(X_test)))

print(f'Poly(degree=2, all features) RMSE — Train: {rmse_p2_all_train:.4f}')
print(f'Poly(degree=2, all features) RMSE — Test : {rmse_p2_all_test:.4f}')

In [ ]:
# ============================================================
# Q6: KNN Regression — k = 5
#
# Fit a KNN regression with n_neighbors=5 using ALL predictors.
#
# QUESTION: What is the RMSE in the TRAIN sample?
# ANSWER  : 14.2940
#
# KEY CONCEPT: KNN regression predicts using the mean rating
# of the 5 nearest neighbors. With k=5 the model is flexible
# (low bias, higher variance) → low train RMSE but may overfit.
# ============================================================

knn5 = KNeighborsRegressor(n_neighbors=5)
knn5.fit(X_train, y_train)

rmse_knn5_train = np.sqrt(mean_squared_error(y_train, knn5.predict(X_train)))
rmse_knn5_test  = np.sqrt(mean_squared_error(y_test,  knn5.predict(X_test)))

print(f'KNN(k=5) RMSE — Train: {rmse_knn5_train:.4f}')
print(f'KNN(k=5) RMSE — Test : {rmse_knn5_test:.4f}')

In [ ]:
# ============================================================
# Q7: KNN Regression — k = 20
#
# Fit a KNN regression with n_neighbors=20 using ALL predictors.
#
# QUESTION: What is the RMSE in the TRAIN sample?
# ANSWER  : 15.6406
#
# KEY INSIGHT: Larger k → smoother predictions → higher bias,
# lower variance. Compare:
#   k=5  → Train RMSE 14.29 / Test RMSE 17.70  (overfits)
#   k=20 → Train RMSE 15.64 / Test RMSE 16.50  (generalises better)
# ============================================================

knn20 = KNeighborsRegressor(n_neighbors=20)
knn20.fit(X_train, y_train)

rmse_knn20_train = np.sqrt(mean_squared_error(y_train, knn20.predict(X_train)))
rmse_knn20_test  = np.sqrt(mean_squared_error(y_test,  knn20.predict(X_test)))

print(f'KNN(k=20) RMSE — Train: {rmse_knn20_train:.4f}')
print(f'KNN(k=20) RMSE — Test : {rmse_knn20_test:.4f}')

In [ ]:
# ============================================================
# Q8: Bias–Variance Tradeoff Summary
#
# QUESTION A: Which model shows the clearest sign of OVERFITTING
#             (large gap between train and test RMSE)?
# ANSWER A  : KNN(k=5) — Train 14.29 vs Test 17.70 (gap of 3.41)
#
# QUESTION B: Which model achieves the LOWEST test RMSE overall?
# ANSWER B  : Poly(degree=2, all features) with Test RMSE 15.2972
#
# KEY CONCEPT: The bias-variance tradeoff:
#   High flexibility (small k, high degree) → low bias, high variance
#   Low flexibility (large k, low degree)   → high bias, low variance
# Optimal models balance the two.
# ============================================================

summary = {
    'LR (loudness)':           (16.2925, 16.2753),
    'Poly(2, loudness)':       (16.2617, 16.2299),
    'Poly(5, loudness)':       (16.2354, 16.2143),
    'Poly(2, all features)':   (15.2410, 15.2972),
    'KNN(k=5)':                (14.2940, 17.6960),
    'KNN(k=20)':               (15.6406, 16.5011),
}

df_summary = pd.DataFrame(summary, index=['Train RMSE', 'Test RMSE']).T
df_summary['Gap (Test-Train)'] = df_summary['Test RMSE'] - df_summary['Train RMSE']
df_summary = df_summary.sort_values('Test RMSE')
print(df_summary.to_string())